In [ ]:
# Resolve project root (Colab Drive mount, else local repo). Verifies src/; fails loud if missing.
import sys
from pathlib import Path

ROOT = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    !pip install -q pyspark mlflow seaborn pyyaml
    # First lab3-data-engineering under MyDrive that actually contains src/.
    for cand in Path('/content/drive/MyDrive').rglob('lab3-data-engineering'):
        if (cand / 'src').is_dir():
            ROOT = cand
            break
except ImportError:
    # Local: walk up from CWD to the repo root (works from notebooks/ or repo root).
    ROOT = next((b for b in (Path.cwd(), *Path.cwd().parents)
                 if (b / 'src').is_dir() and (b / 'config' / 'config.yaml').exists()), None)

if ROOT is None or not (ROOT / 'src').is_dir():
    md = Path('/content/drive/MyDrive')
    where = [p.name for p in md.iterdir()] if md.exists() else 'Drive not mounted'
    raise FileNotFoundError(f"lab3-data-engineering with src/ not found (ROOT={ROOT}). MyDrive top level: {where}")

ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

In [ ]:
import shutil

import mlflow
import mlflow.spark
import pandas as pd

from src.utils.config import load_config
from src.utils.spark_session import get_spark
from src import visuals
from src.training import (
    METRICS, SEED, load_training_df, confusion_matrix, label_order, per_class_metrics,
)

config = load_config()

## B.3 — Results

Reads the runs logged by `training.ipynb` and the promoted model from the registry.
Restores the MLflow store from Drive so artifact paths resolve, then charts model
comparison and the best model's confusion matrix / per-class metrics.

In [ ]:
# Restore the MLflow store from Drive to local (artifacts are referenced by local path).
drive_store = ROOT / "mlflow"
local_store = Path("/content/mlflow") if Path("/content").exists() else drive_store
if local_store != drive_store and drive_store.exists() and not local_store.exists():
    shutil.copytree(drive_store, local_store)

mlflow.set_tracking_uri(f"sqlite:///{local_store / 'mlflow.db'}")
experiment = config["mlflow"]["experiment"]

### Model comparison (validation metrics)

In [ ]:
# Latest run per model (re-running training appends new runs).
runs = mlflow.search_runs(experiment_names=[experiment],
                          order_by=["attributes.start_time DESC"])
comp = runs.rename(columns={f"metrics.{m}": m for m in METRICS})
comp["model"] = runs["tags.mlflow.runName"]
comp = (comp.dropna(subset=["model"])
        .drop_duplicates("model", keep="first")[["model", *METRICS]]
        .sort_values("accuracy", ascending=False)
        .reset_index(drop=True))
comp

In [ ]:
visuals.plot_model_comparison(comp, "accuracy")
visuals.plot_metric_comparison(comp, METRICS)

### Best model: confusion matrix + per-class metrics

Loads the promoted model from the registry and re-scores the same validation split
(reproducibility check), then shows per-tier performance.

In [ ]:
model_name = config["analysis"]["model_name"]
# Create the configured Spark session BEFORE load_model (which would otherwise
# auto-create a default one and the config would be ignored).
spark = get_spark(config)
try:
    model = mlflow.spark.load_model(f"models:/{model_name}/Production")
except Exception:
    model = mlflow.spark.load_model(f"models:/{model_name}@production")

# Same sort + seed + fraction as train(), so this is the exact validation split.
df = load_training_df(spark, config)
df = df.orderBy(*df.columns)
frac = config["analysis"].get("train_fraction", 0.8)
_, val_df = df.randomSplit([frac, 1 - frac], seed=SEED)

preds = model.transform(val_df)
order = label_order(model)
visuals.plot_confusion_matrix(confusion_matrix(preds, order), order, model_name)
pd.Series(per_class_metrics(preds, order)).round(3)